In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import warnings
import io
import requests

# Desactivar advertencias de pandas
warnings.filterwarnings('ignore')

# --- CONFIGURACIÓN DE PARÁMETROS ---

# 1. FUENTE DE DATOS
URL_CSV = "https://datos.madrid.es/egob/catalogo/212411-44-madrid-avisa.csv"
SEPARADOR_CSV = ';'
CATEGORIA_FILTRO = 'Arboles y parques' 
NOMBRE_COLUMNA_DISTRITO = 'DISTRITO' 
# 💡 NUEVO PARÁMETRO: Umbral mínimo de registros para incluir el distrito en el análisis
UMBRAL_MINIMO_REGISTROS = 500 

# 2. DATOS DE POBLACIÓN DE MADRID (CENSO 2024 - ESTIMADO)
# Nombres de distritos sin tilde para coincidir con la limpieza del CSV.
POBLACION_DISTRITOS = {
    'CENTRO': 149692, 'ARGANZUELA': 156801, 'RETIRO': 120516, 
    'SALAMANCA': 147775, 'CHAMARTIN': 146193, 'TETUAN': 161823, 
    'CHAMBERI': 139199, 'FUENCARRAL-EL PARDO': 250198, 'MONCLOA-ARAVACA': 137887,
    'LATINA': 239088, 'CARABANCHEL': 264903, 'USERA': 144572, 
    'PUENTE DE VALLECAS': 244584, 'MORATALAZ': 94656, 'CIUDAD LINEAL': 214734,
    'HORTALEZA': 195768, 'VILLAVERDE': 100742, 'VILLA DE VALLECAS': 122824, 
    'VICALVARO': 87588, # CORREGIDO sin tilde
    'SAN BLAS-CANILLEJAS': 158363, 'BARAJAS': 49454 
}

# ----------------------------------------------------------------------
# FUNCIONES AUXILIARES
# ----------------------------------------------------------------------

def cargar_datos_desde_url(url, sep, parse_cols, dayfirst):
    """Descarga, carga y parsea el CSV desde la URL."""
    try:
        response = requests.get(url)
        response.raise_for_status()
        try:
            df = pd.read_csv(io.StringIO(response.content.decode('utf-8')), sep=sep, parse_dates=parse_cols, dayfirst=dayfirst)
        except UnicodeDecodeError:
            df = pd.read_csv(io.StringIO(response.content.decode('latin1')), sep=sep, parse_dates=parse_cols, dayfirst=dayfirst)
        return df
    except requests.exceptions.RequestException as e:
        print(f"❌ ERROR al descargar la URL: {e}")
        return pd.DataFrame()

def generar_graficos(df_analisis, meses_filtro):
    """Genera los tres gráficos y sus tablas separadas."""
    
    meses_titulo = ', '.join(map(str, meses_filtro))
    
    tabla_resumen = df_analisis.rename(columns={
        'Incidencia_1000_Hab': 'Incidencias/1000 Hab.', 
        'Media_Dias_Respuesta': 'Media Días Resp.',
        'POBLACION_CENSO': 'Población'
    }).set_index(NOMBRE_COLUMNA_DISTRITO)[['Registros', 'Incidencias/1000 Hab.', 'Media Días Resp.', 'Población']]
    
    
    # --- GRÁFICO 1: Registros Absolutos ---
    plt.figure(figsize=(14, 6))
    sns.barplot(
        data=df_analisis.sort_values(by='Registros', ascending=False),
        x=NOMBRE_COLUMNA_DISTRITO, y='Registros', palette='Blues_d'
    )
    plt.title(f'1. Número de Registros de "{CATEGORIA_FILTRO}" por Distrito (Meses: {meses_titulo})', fontsize=14)
    plt.xlabel('Distrito')
    plt.ylabel('Nº de Registros (Incidencias Absolutas)')
    plt.xticks(rotation=45, ha='right')
    plt.grid(axis='y', linestyle='--', alpha=0.7)
    plt.tight_layout()
    plt.show()
    
    # --- TABLA 1 (Separada) ---
    print("\n" + "="*50)
    print(f"📊 TABLA 1: DATOS COMPLETOS DE REGISTROS Y POBLACIÓN ({CATEGORIA_FILTRO})")
    print(f"Nota: Solo distritos con más de {UMBRAL_MINIMO_REGISTROS} registros.")
    print("="*50)
    print(tabla_resumen)
    print("\n")

    # ----------------------------------------------------------

    # --- GRÁFICO 2: Incidencia Normalizada por 1,000 Habitantes ---
    
    plt.figure(figsize=(14, 6))
    sns.barplot(
        data=df_analisis.sort_values(by='Incidencia_1000_Hab', ascending=False),
        x=NOMBRE_COLUMNA_DISTRITO, 
        y='Incidencia_1000_Hab', 
        palette='Reds_d'
    )
    plt.title(f'2. Incidencia Normalizada por 1.000 Habitantes (Meses: {meses_titulo})', fontsize=14)
    plt.xlabel('Distrito')
    plt.ylabel('Incidencias/1000 Hab.') 
    plt.xticks(rotation=45, ha='right')
    plt.grid(axis='y', linestyle='--', alpha=0.7)
    plt.tight_layout()
    plt.show()
    
    # --- TABLA 2 (Separada) ---
    print("\n" + "="*50)
    print(f"📊 TABLA 2: DATOS COMPLETOS DE REGISTROS Y POBLACIÓN ({CATEGORIA_FILTRO})")
    print(f"Nota: Solo distritos con más de {UMBRAL_MINIMO_REGISTROS} registros.")
    print("="*50)
    print(tabla_resumen)
    print("\n")

    # ----------------------------------------------------------

    # --- GRÁFICO 3: Tiempo Medio de Respuesta por Distrito ---
    plt.figure(figsize=(14, 6))
    sns.barplot(
        data=df_analisis.sort_values(by='Media_Dias_Respuesta', ascending=False),
        x=NOMBRE_COLUMNA_DISTRITO, y='Media_Dias_Respuesta', palette='viridis'
    )
    plt.title(f'3. Tiempo Medio de Respuesta por Distrito (Días) (Meses: {meses_titulo})', fontsize=14)
    plt.xlabel('Distrito')
    plt.ylabel('Media de Días de Respuesta')
    plt.xticks(rotation=45, ha='right')
    plt.grid(axis='y', linestyle='--', alpha=0.7)
    plt.tight_layout()
    plt.show()
    
    # --- TABLA 3 (Separada) ---
    print("\n" + "="*50)
    print(f"📊 TABLA 3: DATOS COMPLETOS DE REGISTROS Y POBLACIÓN ({CATEGORIA_FILTRO})")
    print(f"Nota: Solo distritos con más de {UMBRAL_MINIMO_REGISTROS} registros.")
    print("="*50)
    print(tabla_resumen)
    print("\n")


# ----------------------------------------------------------------------
## 3. Lógica Principal con Filtro de Umbral
# ----------------------------------------------------------------------

def preparar_y_analizar_datos():
    """Ejecuta la carga, preparación y aplica el filtro de umbral."""
    
    print("Iniciando descarga y análisis de datos de Madrid...")
    df = cargar_datos_desde_url(URL_CSV, SEPARADOR_CSV, ['FECHA_DE_RECEPCION', 'FECHA_RESOLUCION'], True)

    if df.empty: return
    
    # Limpieza estricta del Distrito (MAYÚSCULAS y sin espacios)
    df[NOMBRE_COLUMNA_DISTRITO] = df[NOMBRE_COLUMNA_DISTRITO].str.strip().str.upper()

    # Diagnóstico de Categorías
    categorias_disponibles = df['CATEGORIA_NIVEL1'].value_counts().head(10)
    print("\n--- 🔎 Top 10 Categorías disponibles en los datos ---")
    print(f"La categoría actual es: '{CATEGORIA_FILTRO}'. Verifique que coincide:")
    print(categorias_disponibles)
    
    # Pre-procesamiento general y filtro por categoría
    df_filtrado = df[
        (df['CATEGORIA_NIVEL1'] == CATEGORIA_FILTRO) &
        (df[NOMBRE_COLUMNA_DISTRITO].notna())
    ].copy()

    df_filtrado['Dias_Respuesta'] = (df_filtrado['FECHA_RESOLUCION'] - df_filtrado['FECHA_DE_RECEPCION']).dt.days

    # Filtro de Meses Interactivo
    print("\n--- 📆 Filtro Temporal ---")
    meses_str = input("Introduce los meses separados por comas (Ej: 6,7,8 para verano): ")
    try:
        meses_filtro = [int(m.strip()) for m in meses_str.split(',') if m.strip().isdigit() and 1 <= int(m.strip()) <= 12]
        if not meses_filtro: meses_filtro = list(range(1, 13))
    except Exception:
        meses_filtro = list(range(1, 13))
        
    df_final = df_filtrado[df_filtrado['FECHA_DE_RECEPCION'].dt.month.isin(meses_filtro)].copy()
    
    if df_final.empty:
        print(f"⚠️ No hay **registros** para la categoría '{CATEGORIA_FILTRO}' y meses seleccionados.")
        return

    print(f"✅ Análisis ejecutado para la Categoría '{CATEGORIA_FILTRO}' y meses: {meses_filtro}")

    # ----------------------------------------------------------------------
    # 💡 FILTRO DE UMBRAL DE REGISTROS (SOLUCIÓN)
    # ----------------------------------------------------------------------
    
    # 1. Generar estadísticas iniciales (incluyendo el conteo total)
    estadisticas = df_final.groupby(NOMBRE_COLUMNA_DISTRITO).agg(
        Registros=('FECHA_DE_RECEPCION', 'size'),
        Media_Dias_Respuesta=('Dias_Respuesta', 'mean')
    ).reset_index()

    # 2. Aplicar el filtro de umbral
    df_filtrado_umbral = estadisticas[estadisticas['Registros'] >= UMBRAL_MINIMO_REGISTROS].copy()
    
    print(f"\n📢 Se han excluido {len(estadisticas) - len(df_filtrado_umbral)} distrito(s) con menos de {UMBRAL_MINIMO_REGISTROS} registros.")
    
    if df_filtrado_umbral.empty:
        print("⚠️ Tras aplicar el filtro de umbral, no queda ningún distrito para el análisis.")
        return

    # 3. Merge con Población y Cálculo de Incidencia (Solo para los distritos filtrados)
    df_poblacion = pd.DataFrame(list(POBLACION_DISTRITOS.items()), columns=[NOMBRE_COLUMNA_DISTRITO, 'POBLACION_CENSO'])
    df_analisis = pd.merge(df_filtrado_umbral, df_poblacion, on=NOMBRE_COLUMNA_DISTRITO, how='left')
    
    # Diagnóstico y Corrección de Población (si falla el merge)
    df_analisis['POBLACION_CENSO_REAL'] = df_analisis['POBLACION_CENSO'].copy() 
    df_analisis['POBLACION_CENSO'] = df_analisis['POBLACION_CENSO'].fillna(1)
    
    # Cálculo: (Registros / Población) * 1000
    df_analisis['Incidencia_1000_Hab'] = (
        df_analisis['Registros'] / (df_analisis['POBLACION_CENSO'] / 1000)
    ).round(2).replace([float('inf'), -float('inf')], 0).fillna(0)
    
    df_analisis['Media_Dias_Respuesta'] = df_analisis['Media_Dias_Respuesta'].round(2)
    
    # Usamos la población original (o 0 si no se encontró) para la tabla de resumen
    df_analisis['POBLACION_CENSO'] = df_analisis['POBLACION_CENSO_REAL'].fillna(0)
    df_analisis = df_analisis.sort_values(by='Registros', ascending=False)

    # ----------------------------------------------------------------------
    ## 4. Visualizaciones y Tablas Separadas
    # ----------------------------------------------------------------------
    generar_graficos(df_analisis, meses_filtro)


# --- EJECUCIÓN DEL PROGRAMA ---
if __name__ == "__main__":
    preparar_y_analizar_datos()